# Customer Segmentation using Local Models 🚀

โน้ตบุ๊กนี้จะทำการแบ่งกลุ่มลูกค้า (User Segmentation) จากคอมเมนต์ YouTube โดยใช้โมเดล Local ทั้งหมด (ไม่ใช้ API เสียเงิน) 
กระบวนการทำงานมีดังนี้:
1. **Scraping**: ดึงคอมเมนต์จาก YouTube
2. **Embedding**: แปลงข้อความเป็น Vector ด้วยโมเดล Open Source (Sentence Transformers)
3. **Clustering**: จัดกลุ่มด้วย K-Means และ Agglomerative Clustering (พร้อมคำอธิบายตัวแปรละเอียด)
4. **Analysis**: วิเคราะห์และสรุปผลแต่ละกลุ่มด้วย Local LLM

In [ ]:
# ติดตั้ง Library ที่จำเป็น (รันครั้งเดียว)
!pip install youtube-comment-downloader pandas scikit-learn seaborn matplotlib sentence-transformers torch transformers accelerate bitsandbytes langchain-huggingface

## 1. ดึงข้อมูลคอมเมนต์ (Data Scraping) 📥
ใช้ `youtube_comment_downloader` ในการดึงข้อมูลคอมเมนต์จากวิดีโอที่ต้องการ

In [ ]:
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_RECENT
import pandas as pd

def scrape_youtube_comments(video_url, max_comments=300):
    print(f"กำลังดึงคอมเมนต์จาก: {video_url}")
    downloader = YoutubeCommentDownloader()
    generator = downloader.get_comments_from_url(video_url, sort_by=SORT_BY_RECENT)
    
    comments = []
    for count, comment in enumerate(generator):
        if count >= max_comments: break
        comments.append({
            'Author': comment['author'],
            'Comment Text': comment['text']
        })
    
    df = pd.DataFrame(comments)
    df.to_csv('local_youtube_comments.csv', index=False, encoding='utf-8-sig')
    print(f"✅ บันทึก {len(df)} คอมเมนต์เรียบร้อย!")
    return df

# ตัวอย่าง URL (สามารถเปลี่ยนได้)
url = 'https://youtu.be/iogcY_4xGjo?si=yX71k-5LL2baunRC'  # เปลี่ยนเป็น URL ที่ต้องการ
df = scrape_youtube_comments(url, max_comments=200)
df.head()

## 2. สร้าง Embeddings (Text Representation) 🧠
ใช้โมเดล `sentence-transformers/all-MiniLM-L6-v2` หรือโมเดลอื่น ๆ ที่รันบนเครื่อง Local ได้ เพื่อแปลงข้อความเป็น Vector

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle
import os

# ตั้งชื่อไฟล์สำหรับเก็บ Vector Store (Simulation)
VECTOR_STORE_FILE = 'local_vector_store.pkl'

def get_local_embeddings(texts, model_name='all-MiniLM-L6-v2'):
    print(f"🔄 กำลังโหลดโมเดล Embedding: {model_name} ...")
    model = SentenceTransformer(model_name)
    print("🚀 กำลังสร้าง Embeddings...")
    embeddings = model.encode(texts, show_progress_bar=True)
    return embeddings

# ตรวจสอบว่ามีไฟล์ Vector Store อยู่แล้วหรือไม่ เพื่อไม่ต้องคำนวณใหม่
if os.path.exists(VECTOR_STORE_FILE):
    print(f"✅ พบไฟล์ Vector Store: {VECTOR_STORE_FILE} กำลังโหลด...")
    with open(VECTOR_STORE_FILE, 'rb') as f:
        embeddings = pickle.load(f)
else:
    # ถ้ายังไม่มี ให้สร้างใหม่
    texts = df['Comment Text'].tolist()
    embeddings = get_local_embeddings(texts)
    
    # บันทึกเก็บไว้ (Vector Store)
    with open(VECTOR_STORE_FILE, 'wb') as f:
        pickle.dump(embeddings, f)
    print(f"💾 บันทึก Vector Store เรียบร้อยที่: {VECTOR_STORE_FILE}")

print(f"ได้ Embeddings ขนาด: {embeddings.shape}")

## 3. การจัดกลุ่ม (Clustering) ด้วย K-Means 🔢
เราจะนำ Vector ที่ได้มาจัดกลุ่ม โดยใช้ K-Means

### อธิบายตัวแปร (Hyperparameters) ที่สำคัญ:
- **`n_clusters`**: จำนวนกลุ่มที่ต้องการแบ่ง (เช่น 5 กลุ่ม) ยิ่งเยอะยิ่งละเอียด แต่ถ้าเยอะเกินไปอาจจะตีความยาก
- **`random_state`**: ค่า Seed สำหรับการสุ่มเริ่มต้น เพื่อให้ผลลัพธ์เหมือนเดิมทุกครั้งที่รัน (Reproducibility) เช่น ใส่ `42`
- **`n_init`**: จำนวนครั้งที่ Algo จะรันโดยเริ่มสุ่มจุด Centroid ใหม่ในแต่ละรอบ แล้วเลือกผลลัพธ์ที่ดีที่สุด (ค่า default มักจะเป็น 10) ช่วยแก้ปัญหาการติด Local Optima

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# กำหนดค่าพารามิเตอร์
num_clusters = 5  # อยากได้กี่กลุ่มแก้ตรงนี้
seed = 42         # ล็อคผลลัพธ์ให้เหมือนเดิม
n_init_val = 10   # จำนวนรอบการสุ่มเริ่มต้น

print(f"⚙️ เริ่มทำ Clustering: n_clusters={num_clusters}, random_state={seed}, n_init={n_init_val}")

kmeans = KMeans(n_clusters=num_clusters, random_state=seed, n_init=n_init_val)
df['Cluster'] = kmeans.fit_predict(embeddings)

# ดูผลลัพธ์เบื้องต้น
print(df['Cluster'].value_counts())

# --- Visualization (Optional) ---
# ลดมิติข้อมูลเหลือ 2D เพื่อพลอตกราฟด้วย PCA
pca = PCA(n_components=2)
reduced_vectors = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=reduced_vectors[:,0], y=reduced_vectors[:,1], hue=df['Cluster'], palette='viridis', style=df['Cluster'])
plt.title('K-Means Clustering Visualization (PCA)')
plt.show()

## 4. เพิ่มประสิทธิภาพด้วย Agglomerative Clustering (Hierarchical) 🌳
เพิ่มโมเดลทางเลือกเพื่อเปรียบเทียบผลลัพธ์ และใช้ Silhouette Score วัดประสิทธิภาพความแม่นยำ

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

print("🚀 กำลังเพิ่มโมเดล Agglomerative Clustering (Hierarchical) เพื่อเปรียบเทียบ...")

# 1. Agglomerative Clustering
# โมเดลนี้จะสร้างกลุ่มแบบลำดับชั้น (Hierarchy) เหมาะสำหรับข้อมูลที่มีโครงสร้างซับซ้อนกว่า K-Means
agg_clustering = AgglomerativeClustering(n_clusters=num_clusters)
df['Cluster_Agg'] = agg_clustering.fit_predict(embeddings)

# 2. เปรียบเทียบประสิทธิภาพด้วย Silhouette Score
# ค่า Silhouette Score ยิ่งสูง (ใกล้ 1) ยิ่งดี แสดงว่ากลุ่มแยกออกจากกันได้ชัดเจน
kmeans_score = silhouette_score(embeddings, df['Cluster'])
agg_score = silhouette_score(embeddings, df['Cluster_Agg'])

print(f"📊 ผลการเปรียบเทียบ Silhouette Score (ยิ่งสูงยิ่งดี):")
print(f"   - K-Means Score: {kmeans_score:.4f}")
print(f"   - Agglomerative Score: {agg_score:.4f}")

# เลือกโมเดลที่ดีที่สุด
if agg_score > kmeans_score:
    print("✅ แนะนำ: Agglomerative Clustering ให้ผลลัพธ์ที่ดีกว่าในการแยกกลุ่มนี้")
    best_cluster_col = 'Cluster_Agg'
else:
    print("✅ แนะนำ: K-Means ยังคงให้ผลลัพธ์ที่ดีกว่า (หรือใกล้เคียงกัน)")
    best_cluster_col = 'Cluster'

# Visualization เปรียบเทียบ 2 โมเดล
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot K-Means
sns.scatterplot(x=reduced_vectors[:,0], y=reduced_vectors[:,1], hue=df['Cluster'], palette='viridis', style=df['Cluster'], ax=axes[0])
axes[0].set_title(f'K-Means (Score: {kmeans_score:.3f})')

# Plot Agglomerative
sns.scatterplot(x=reduced_vectors[:,0], y=reduced_vectors[:,1], hue=df['Cluster_Agg'], palette='viridis', style=df['Cluster_Agg'], ax=axes[1])
axes[1].set_title(f'Agglomerative (Score: {agg_score:.3f})')

plt.show()

## 5. วิเคราะห์และสรุปผลด้วย Local LLM 🤖
เราจะใช้โมเดลภาษาขนาดใหญ่ที่รันบนเครื่อง (Local LLM) เพื่อสรุปว่าแต่ละกลุ่มคือใคร มีลักษณะนิสัยอย่างไร
โดยเลือกผลลัพธ์จากโมเดลที่มีค่า Silhouette Score สูงที่สุด

In [ ]:
import sys
import os

# เพิ่ม path ให้ Python มองเห็นไฟล์ local_llm.py ในโฟลเดอร์เดียวกัน
sys.path.append(os.getcwd())

# Import ฟังก์ชันโหลดโมเดลจากไฟล์ local_llm.py ที่มีอยู่แล้ว
# (ตรวจสอบให้แน่ใจว่าไฟล์ local_llm.py อยู่ในโฟลเดอร์เดียวกับ Notebook นี้)
try:
    from local_llm import load_local_chat_model
    # ใช้โมเดลขนาดเล็กหน่อยเพื่อความรวดเร็วในการเทส (หรือใช้ 7B ตามเดิมก็ได้ถ้าเครื่องไหว)
    # แนะนำ Qwen/Qwen2.5-1.5B-Instruct สำหรับการทดสอบเร็วๆ แต่ถ้าเอาฉลาดๆใช้ 7B
    chat_model = load_local_chat_model(model_name="Qwen/Qwen2.5-1.5B-Instruct") 
except ImportError:
    print("❌ ไม่พบไฟล์ local_llm.py หรือ library ไม่ครบ")
    # Fallback หรือแจ้งเตือนให้User ตรวจสอบ

from langchain_core.messages import HumanMessage, SystemMessage

def analyze_cluster(cluster_id, sample_comments):
    prompt = f"""
    คุณคือนักวิเคราะห์ข้อมูลลูกค้า ฉันมีกลุ่มคอมเมนต์จากลูกค้ากลุ่มหนึ่ง 
    ช่วยวิเคราะห์และสรุป Persona ของลูกค้ากลุ่มนี้ให้หน่อย โดยพิจารณาจากคอมเมนต์ตัวอย่างเหล่านี้:
    
    {sample_comments}
    
    ให้สรุปเป็นหัวข้อดังนี้:
    1. ชื่อกลุ่ม (ตั้งชื่อเท่ๆ ให้กลุ่มนี้)
    2. นิสัย/ความชอบ
    3. อารมณ์ (Positive/Negative/Neutral)
    """
    
    messages = [
        SystemMessage(content="You are a helpful data analyst assistant."),
        HumanMessage(content=prompt)
    ]
    
    response = chat_model.invoke(messages)
    return response.content

# ตรวจสอบตัวแปร best_cluster_col ถ้าไม่มีให้ใช้ Cluster ปกติ
target_column = best_cluster_col if 'best_cluster_col' in locals() else 'Cluster'
print(f"🔥 กำลังวิเคราะห์โดยใช้ผลลัพธ์จาก: {target_column}")

# วนลูปวิเคราะห์ทีละกลุ่ม
for i in range(num_clusters):
    print(f"\n🔹 กำลังวิเคราะห์กลุ่มที่ {i}...")
    
    # สุ่มตัวอย่างคอมเมนต์ในกลุ่มมา 10-15 คอมเมนต์ เพื่อไม่ให้ Token ยาวเกินไป
    cluster_comments = df[df[target_column] == i]['Comment Text'].sample(n=min(10, len(df[df[target_column] == i]))).tolist()
    comments_str = "\n- ".join(cluster_comments)
    
    summary = analyze_cluster(i, comments_str)
    print(summary)
    print("-"*50)